In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [3]:
# Read and prepare the data
data = pd.read_csv("/home/rachel/Desktop/data/clean_data_all.csv")

# Extract numeric id 
data['id'] = data['id'].astype(str).str.replace("sub-", "", regex=True).astype(float)

# Create cohort variable
data['cohort'] = np.where(data['id'] > 5000, "bbhi", "bbhi_senior")

# Pivot data from wide to long 
pattern = re.compile(r"(.+)_(\d+)$") # Identify columns with timepoints
columns_to_pivot = [col for col in data.columns if pattern.match(col)]

# Get unique base variable names
base_variables = set()
for col in columns_to_pivot:
    match = pattern.match(col)
    if match:
        base_variables.add(match.group(1))

# Identify the non-timepoint columns 
id_columns = [col for col in data.columns if col not in columns_to_pivot]

# Create an empty list to store dataframes for each timepoint
timepoint_dfs = []

# For each timepoint, create a dataframe with all variables for that timepoint
timepoints = sorted(set([int(pattern.match(col).group(2)) for col in columns_to_pivot if pattern.match(col)]))

for tp in timepoints:
    # Create a new dataframe with just the non-timepoint columns
    tp_data = data[id_columns].copy()
    
    # Add a timepoint column
    tp_data['timepoint'] = tp
    
    # For each base variable, add its value at the current timepoint
    for base_var in base_variables:
        timepoint_col = f"{base_var}_{tp}"
        if timepoint_col in data.columns:
            tp_data[base_var] = data[timepoint_col]
    
    # Add this to the list
    timepoint_dfs.append(tp_data)

# Combine all timepoint dataframes 
long_data = pd.concat(timepoint_dfs, ignore_index=True)

In [ ]:
def run_models(df, independent_vars, dependent_vars, timepoint=None, interaction=None, model_type="lm"):
    """
    Runs a linear model or ANOVA for the given variables and returns the p-values and F-values.

    Args:
        df: dataframe with the data
        independent_vars: list of columns to treat as independent variables
        dependent_vars: list of columns to treat as dependent variables 
        timepoint: timepoint (1 or 2, defaults to 1 if None (e.g. uses baseline age for longitudinal models))
        interaction: variable to interact with independent variables (optional)
        model_type: "aov" for ANOVA or "lm" for linear model 
        
    Returns:
        DataFrame with model results including FDR-corrected p-values
    """
    all_results = []
    
    for ivar in independent_vars:
        for dvar in dependent_vars:
            # Skip if dependent variable is not in the dataframe
            if dvar not in df.columns:
                continue
                
            # Determine age variable based on timepoint
            age_var = "age_2" if timepoint == 2 else "age_1"
            
            # Create formula string based on parameters
            if interaction is None:
                formula_str = f'scale({dvar}) ~ C(scale({ivar})) + scale({age_var}) + scale(YoE) + sex'
            else:
                formula_str = f'scale({dvar}) ~ C(scale({ivar})) * scale({interaction}) + scale({age_var}) + scale(YoE) + sex'
            
            try:
                # Fit model using OLS (for both ANOVA and LM)
                model = smf.ols(formula_str, data=df)
                fit_results = model.fit()
                
                # Get adjusted R-squared (same for both model types)
                adj_r2 = fit_results.rsquared_adj
                
                if model_type == "aov":
                    # Create ANOVA table from OLS model
                    aov_table = anova_lm(fit_results, typ=2)
                    
                    # Extract row associated with independent variable
                    search_key = f'C(scale({ivar}))'
                    if search_key in aov_table.index:
                        F_val = aov_table.loc[search_key, 'F']
                        p_val = aov_table.loc[search_key, 'PR(>F)']
                        beta = None
                        ci_lower = None
                        ci_upper = None
                    else:
                        continue
                        
                elif model_type == "lm":
                    # For linear models, extract coefficient info
                    relevant_coefs = [coef for coef in fit_results.params.index if coef.startswith(f'C(scale({ivar}))')]
                    if not relevant_coefs:
                        continue
                        
                    # Get first coefficient info 
                    coef_key = relevant_coefs[0]
                    beta = fit_results.params[coef_key]
                    p_val = fit_results.pvalues[coef_key]
                    F_val = None
                    
                    # Get confidence interval
                    ci = fit_results.conf_int()
                    ci_lower = ci.loc[coef_key, 0]  # Lower bound
                    ci_upper = ci.loc[coef_key, 1]  # Upper bound

                # Add significance flag
                sig = (p_val < 0.05)
                
                # Append results with all requested metrics
                all_results.append({
                    'dependent_var': dvar,
                    'independent_var': ivar,
                    'F_value': F_val,
                    'p_value': p_val,
                    'beta': beta,
                    'ci_lower': ci_lower if model_type == "lm" else None,
                    'ci_upper': ci_upper if model_type == "lm" else None,
                    'adj_r2': adj_r2,
                    'significant': sig
                })
                
            except Exception as e:
                continue
        
    # If no results were found, return an empty DataFrame with the expected columns
    if not all_results:
        return pd.DataFrame(columns=['dependent_var', 'independent_var', 'F_value', 
                                    'p_value', 'significant', 'fdr_p_value', 'fdr_significant'])
    
    # Create results DataFrame
    results_df = pd.DataFrame(all_results, columns=['dependent_var', 'independent_var', 
                                                  'F_value', 'p_value', 'significant'])
    
    # Sort by p-value
    results_df = results_df.sort_values('p_value')
    
    # Apply FDR correction across all tests
    from statsmodels.stats.multitest import multipletests
    if len(results_df) > 0:
        rejected, fdr_p, _, _ = multipletests(results_df['p_value'], method='fdr_bh')
        results_df['fdr_p_value'] = fdr_p
        results_df['fdr_significant'] = rejected
    
    return results_df

# Test model
vars_long_sfc = [
    "sfc_all_slopes",
    "sfc_Default_slopes",
    "sfc_Frontoparietal_slopes",
    "sfc_VentralAttention_slopes",
    "sfc_DorsalAttention_slopes"
]

lm_results = run_models(data, vars_long_sfc, "memory_slopes", timepoint=None, model_type="lm")
print(lm_results)

Empty DataFrame
Columns: [dependent_var, independent_var, F_value, p_value, significant]
Index: []
